##### Imports and config

In [6]:
import requests
import json
from datetime import datetime, timedelta
from pyspark.sql import functions as F
from pyspark.sql.types import *

# ── Config ──────────────────────────────────────────
FHIR_BASE_URL = "https://hapi.fhir.org/baseR4"
RESOURCES     = ["Patient", "Encounter", "Observation", "Condition"]
PAGE_SIZE     = 50
INGEST_DAYS   = 3
# ────────────────────────────────────────────────────

print("Config loaded")
print(f"Resources   : {RESOURCES}")
print(f"Ingest days : {INGEST_DAYS}")

StatementMeta(, 5d90485d-de9e-4a98-aa29-e642886a7996, 8, Finished, Available, Finished, False)

Config loaded
Resources   : ['Patient', 'Encounter', 'Observation', 'Condition']
Ingest days : 3


##### Functions

In [7]:
def fetch_resource_pages(resource: str, date_str: str) -> list:
    """Fetch all paginated records for a resource on a given date."""
    records  = []
    url      = f"{FHIR_BASE_URL}/{resource}"
    params   = {
        "_count":      PAGE_SIZE,
        "_lastUpdated": f"ge{date_str}",
        "_format":     "json"
    }
    page_num = 0

    while url:
        resp = requests.get(
            url,
            params=params if page_num == 0 else None,
            timeout=60
        )
        resp.raise_for_status()
        bundle  = resp.json()
        entries = bundle.get("entry", [])
        records.extend(entries)
        print(f"    Page {page_num + 1}: {len(entries)} records")

        url = None
        for link in bundle.get("link", []):
            if link["relation"] == "next":
                url = link["url"]
                break

        page_num += 1
        if page_num >= 5:
            print("    Page cap reached.")
            break

    return records

print("Fetch function ready")

StatementMeta(, 5d90485d-de9e-4a98-aa29-e642886a7996, 9, Finished, Available, Finished, False)

Fetch function ready


In [8]:
from notebookutils import mssparkutils

extraction_ts = datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%S")

schema = StructType([
    StructField("resource_type",       StringType(), True),
    StructField("resource_id",         StringType(), True),
    StructField("raw_json",            StringType(), True),
    StructField("extraction_timestamp",StringType(), True),
    StructField("api_url",             StringType(), True),
    StructField("load_date",           StringType(), True),
])

for resource in RESOURCES:
    print(f"\n{'='*45}")
    print(f"Resource: {resource}")

    for day_offset in range(INGEST_DAYS):
        date_str = (datetime.utcnow() - timedelta(days=day_offset)).strftime("%Y-%m-%d")
        print(f"\n  Date: {date_str}")

        try:
            records = fetch_resource_pages(resource, date_str)
        except Exception as e:
            print(f"  ERROR: {e}")
            continue

        if not records:
            print("  No records, skipping.")
            continue

        # ── 1. Save raw JSON to Files layer ──────────
        raw_path = f"Files/raw/{resource.lower()}/{date_str}/response.json"
        mssparkutils.fs.put(
            raw_path,
            json.dumps(records, indent=2),
            overwrite=True
        )
        print(f"  Raw JSON saved → {raw_path}")

        # ── 2. Build rows ─────────────────────────────
        rows = []
        for entry in records:
            res_obj = entry.get("resource", entry)
            rows.append((
                resource,
                res_obj.get("id", "unknown"),
                json.dumps(res_obj),
                extraction_ts,
                f"{FHIR_BASE_URL}/{resource}",
                date_str
            ))

        df = spark.createDataFrame(rows, schema)

        # ── 3. Append to bronze schema Delta table ────
        bronze_table = f"bronze.{resource.lower()}"
        df.write.format("delta") \
          .mode("append") \
          .partitionBy("load_date") \
          .option("mergeSchema", "true") \
          .saveAsTable(bronze_table)

        print(f"  Appended {len(rows)} rows → {bronze_table}")

print("\nBronze ingestion complete!")

StatementMeta(, 5d90485d-de9e-4a98-aa29-e642886a7996, 10, Finished, Available, Finished, False)


Resource: Patient

  Date: 2026-06-16
    Page 1: 50 records
    Page 2: 50 records
    Page 3: 50 records
    Page 4: 50 records
    Page 5: 50 records
    Page cap reached.
  Raw JSON saved → Files/raw/patient/2026-06-16/response.json
  Appended 250 rows → bronze.patient

  Date: 2026-06-15
    Page 1: 50 records
    Page 2: 50 records
    Page 3: 50 records
    Page 4: 50 records
    Page 5: 50 records
    Page cap reached.
  Raw JSON saved → Files/raw/patient/2026-06-15/response.json
  Appended 250 rows → bronze.patient

  Date: 2026-06-14
    Page 1: 50 records
    Page 2: 50 records
    Page 3: 50 records
    Page 4: 50 records
    Page 5: 50 records
    Page cap reached.
  Raw JSON saved → Files/raw/patient/2026-06-14/response.json
  Appended 250 rows → bronze.patient

Resource: Encounter

  Date: 2026-06-16
    Page 1: 50 records
    Page 2: 50 records
    Page 3: 50 records
    Page 4: 11 records
  Raw JSON saved → Files/raw/encounter/2026-06-16/response.json
  Appended 161 r

In [9]:
print("=== Bronze verification ===\n")

for resource in RESOURCES:
    table = f"bronze.{resource.lower()}"
    count = spark.table(table).count()
    print(f"  {table}: {count} rows")

print("\nSample from bronze.patient:")
spark.table("bronze.patient").select(
    "resource_id", "extraction_timestamp", "load_date"
).show(3)

StatementMeta(, 5d90485d-de9e-4a98-aa29-e642886a7996, 11, Finished, Available, Finished, False)

=== Bronze verification ===

  bronze.patient: 750 rows
  bronze.encounter: 631 rows
  bronze.observation: 750 rows
  bronze.condition: 750 rows

Sample from bronze.patient:
+-----------+--------------------+----------+
|resource_id|extraction_timestamp| load_date|
+-----------+--------------------+----------+
|  136993292| 2026-06-16T16:34:20|2026-06-16|
|  136993293| 2026-06-16T16:34:20|2026-06-16|
|  136993294| 2026-06-16T16:34:20|2026-06-16|
+-----------+--------------------+----------+
only showing top 3 rows

